In [ ]:
# 1. IMPORTS & DRIVE MOUNT
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from google.colab import drive
drive.mount('/content/drive')


# 2. DATASET PATH
data_dir = '/content/drive/MyDrive/224'  # main folder


# 3. BASIC SETTINGS
image_size = (128, 128)
batch_size = 32


# 4. EDA - CLASS DISTRIBUTION

classes = os.listdir(data_dir)
class_counts = []

for cls in classes:
    cls_path = os.path.join(data_dir, cls)
    if os.path.isdir(cls_path):
        class_counts.append(len(os.listdir(cls_path)))

plt.figure(figsize=(8,5))
plt.bar(classes, class_counts, color='skyblue')
plt.title("Class Distribution")
plt.xlabel("Classes")
plt.ylabel("Number of Images")
plt.show()

for cls, count in zip(classes, class_counts):
    print(f"{cls}: {count}")


# 5. SHOW SAMPLE IMAGES

plt.figure(figsize=(12,8))

for i, cls in enumerate(classes):
    cls_path = os.path.join(data_dir, cls)
    img_name = random.choice(os.listdir(cls_path))
    img_path = os.path.join(cls_path, img_name)

    img = load_img(img_path, target_size=image_size)

    plt.subplot(1, len(classes), i+1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis('off')

plt.suptitle("Sample Images")
plt.show()


# 6. IMAGE SIZE DISTRIBUTION
widths, heights = [], []

for cls in classes:
    cls_path = os.path.join(data_dir, cls)
    for img_name in os.listdir(cls_path)[:50]:
        img_path = os.path.join(cls_path, img_name)
        img = Image.open(img_path)
        w, h = img.size
        widths.append(w)
        heights.append(h)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.hist(widths, bins=20)
plt.title("Width Distribution")

plt.subplot(1,2,2)
plt.hist(heights, bins=20)
plt.title("Height Distribution")

plt.show()


# 7. DATA GENERATORS (TRAIN/VAL/TEST SPLIT)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,  # 80% train, 20% val
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# For TEST set, create a separate folder manually:
# /224_test/class1, class2, class3

test_dir = '/content/drive/MyDrive/224_test'

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


# 8. AUGMENTATION VISUALIZATION
sample_batch, _ = next(train_generator)

plt.figure(figsize=(10,10))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(sample_batch[i])
    plt.axis('off')

plt.suptitle("Augmented Images")
plt.show()


# 9. MODEL
def create_model():
    model = models.Sequential()

    model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
    model.add(layers.MaxPooling2D(2,2))

    model.add(layers.Conv2D(64, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D(2,2))

    model.add(layers.Flatten())

    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(3, activation='softmax'))

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

model = create_model()
model.summary()


# 10. CALLBACKS
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]


# 11. TRAINING
history = model.fit(
    train_generator,
    epochs=15,
    validation_data=validation_generator,
    callbacks=callbacks
)


# 12. TRAINING GRAPHS
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Loss")
plt.show()


# 13. TEST EVALUATION
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.4f}")


# 14. CONFUSION MATRIX
y_true = test_generator.classes
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)

disp = ConfusionMatrixDisplay(cm, display_labels=list(test_generator.class_indices.keys()))
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()


# 15. SINGLE IMAGE PREDICTION
img_path = '/content/drive/MyDrive/123.jpg'

img = load_img(img_path, target_size=image_size)
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

prediction = model.predict(img_array)
class_names = list(train_generator.class_indices.keys())

print("Predicted:", class_names[np.argmax(prediction)])